# Исходный код

## Импорты

In [48]:
from IPython.display import clear_output
from datetime import datetime as dt
import random as rnd
import time

## Вспомогательные функции

In [49]:
def choice_prob(more_prob, less_prob, prob=0.8):
    if more_prob and less_prob:
        weights = [prob / len(more_prob)] * len(more_prob) + [(1 - prob) / len(less_prob)] * len(less_prob)
    elif less_prob:
        weights = [1 / len(less_prob)] * len(less_prob)
    else:
        weights = [1 / len(more_prob)] * len(more_prob)

    return rnd.choices(more_prob + less_prob, weights=weights, k=1)[0]

def die_pred(current, max_stat):
    if current >= 2 * max_stat:
        return True
    death_chance = (current / max_stat) ** 2
    return rnd.random() < death_chance

def sex_choice():
    while True:
        yield "XX"
        yield "XY"

sex_choiser = sex_choice()
sex_gen = lambda: next(sex_choiser)

crusade = lambda x, y: [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]

circle = lambda x, y, r: [(i, j) for i in range(x - r, x + r + 1) for j in range(y - r, y + r + 1) if (i, j) != (x, y)]

find_parent_name = lambda obj: obj.__class__.__bases__[0].__name__

find_class_name = lambda obj: obj.__class__.__name__

## Базовые классы

In [50]:
class CelestialObj:
    def __init__(self):
        self.ticks = 0
        self.time = "Day 0 | 0:00 (Night)"
    
    def tick(self):
        self.ticks += 1
        self.time = f"Day {self._get_day()} | {self._get_time()}:00 ({self._get_pos()})"

    def _get_day(self):
        return self.ticks // 24

    def _get_time(self):
        return self.ticks % 24
    
    def _get_pos(self):
        return ["Night", "Morning", "Day", "Afternoon"][((self.ticks % 24 + 1) // 6) % 4]

class World():
    def __init__(self, size, cel_obj=None):
        if cel_obj is None:
            cel_obj = CelestialObj()
            
        self.width = size[0]
        self.height = size[1]
        self.cel_obj = cel_obj
        self.entities = {}
        self.plain = [["|   " for _ in range(self.width + 1)] for _ in range(self.height)]
        self.free_places = [(x, y) for x in range(self.width) for y in range(self.height)]
        
    def add_entity(self, Entity):
        self.entities[Entity.coord] = Entity
        self.free_places.remove(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = Entity.mark
    
    def del_entity(self, Entity):
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "

    def repl_entity(self, New_entity):
        self.entities[New_entity.coord] = New_entity
        x, y = New_entity.coord
        self.plain[y][x] = New_entity.mark
    
    def move_entity(self, Entity, new_place):
        self.entities.pop(Entity.coord)
        self.free_places.append(Entity.coord)
        x, y = Entity.coord
        self.plain[y][x] = "|   "
        self.entities[new_place] = Entity

        if new_place in self.free_places:
            self.free_places.remove(new_place)
            
        x, y = new_place
        self.plain[y][x] = Entity.mark

    def __contains__(self, obj):
        if isinstance(obj, tuple):
            return 0 <= obj[0] < self.width and 0 <= obj[1] < self.height
        return obj in self.entities.values()

class GameLoop:
    def __init__(self, entities_list=[], world_size=(10, 10)):
        self.world = World(world_size)

        for Entity, num in entities_list:
            for _ in range(num):
                place = rnd.choice(self.world.free_places)
                self.world.add_entity(Entity(self.world, place))

    def _tick(self):
        self.world.cel_obj.tick()

        entities = list(self.world.entities.values())
        rnd.shuffle(entities)

        for Entity in entities:
            Entity.live()

    def _render_frame(self):
        time_state = self.world.cel_obj.time
        borders = "...." * self.world.width + "."
        plain = "\n".join(["".join(row) for row in self.world.plain])
        frame = "\n".join([time_state, borders, plain, borders])
        return frame
    
    def _log(self, frame, file):
        with open(f"{file}.txt", "a", encoding="utf-8") as file:
            file.write(frame + "\n")

    def loop(self, ticks=24, delay=0.2, logging=False, log_filename=None, clickable=False):
        if log_filename is None:
            log_filename = dt.now().strftime(r"%H-%M-%S_%d-%m-%Y")

        for _ in range(ticks):
            clear_output(wait=True)

            frame = self._render_frame()
            if logging: self._log(frame, log_filename)
            print(frame)
            self._tick()

            time.sleep(delay)
            if clickable:
                input()

## Сущности

In [51]:
class Entity:
    def __init__(self, world, coord):
        self.world = world
        self.coord = coord

PLANT_REGISTRY = {}
ANIMAL_REGISTRY = {}

class EcosystemMeta(type):
    def __new__(mcs, name, bases, namespace):
        bases_names = [base.__name__ for base in bases]

        cls = super().__new__(mcs, name, bases, namespace)

        if "Plant" in bases_names and name != "Plant":
            PLANT_REGISTRY[name] = cls
        elif "Animal" in bases_names and name != "Animal":
            ANIMAL_REGISTRY[name] = cls

        cls.env_state = 'time'

        cls.mark = ""
        cls.active_time = []
        cls.age = 0

        def _do_nothing(self):
            pass

        def _die(self):
            self.world.del_entity(self)

        def _lifecycle(self):
            pass

        def live(self):
            self._adapt()
            self._lifecycle()

        cls._do_nothing = _do_nothing
        cls._die = _die
        cls._lifecycle = _lifecycle
        cls.live = live

        if "Plant" in bases_names:
            def _adapt(self):
                die = die_pred(self.age, self.max_age)
                self.age += 1
                
                if not (self.world.cel_obj._get_pos() in self.active_time):
                    self._lifecycle = self._die if die else self._do_nothing
                elif die:
                    self._lifecycle = self._die
                else:
                    self._lifecycle = self._grow
                
            def _grow(self):
                if rnd.randint(0, 100) > self.grow_prob:
                    return

                coords = crusade(*self.coord)

                places = []
                prob_places = []

                for coord in coords:
                    if not coord in self.world:
                        continue

                    if coord in self.world.free_places:
                        places.append(coord)
                        continue
                        
                    entity = self.world.entities[coord]
                    same_parent = find_parent_name(entity) == find_parent_name(self)
                    diff_class = find_class_name(entity) != find_class_name(self)
                    
                    if same_parent and diff_class:
                        prob_places.append(coord)

                if places and prob_places:
                    place = choice_prob(places, prob_places, self.repl_prob / 100)
                elif places:
                    place = rnd.choice(places)
                elif prob_places and rnd.randint(0, 100) < self.repl_prob:
                    place = rnd.choice(prob_places)
                else:
                    return
                
                new_plant = self.__class__(self.world, place)

                if place in self.world.free_places:
                    self.world.add_entity(new_plant)
                else:
                    self.world.repl_entity(new_plant)

            cls._adapt = _adapt
            cls._grow = _grow
            
            cls.grow_prob = 60
            cls.repl_prob = 3
            cls.max_age = 48

        elif "Animal" in bases_names:
            def _adapt(self):
                if self.world.cel_obj.ticks == 1:
                    self._lifecycle = self._repr
                    return
                
                self.swarm = self._find_swarm()
                self.aggr = len(self.swarm)
                self.age += 1

                age = die_pred(self.age, self.max_age)
                hunger = die_pred(self.hunger, self.max_hunger)
                die = max(age, hunger)

                if not (self.world.cel_obj._get_pos() in self.active_time):
                    self._lifecycle = self._do_nothing
                elif die:
                    self._lifecycle = self._die
                else:
                    if self.hunger < self.max_hunger * 2 // 3 and self._find_pair() and self.sex == "XX" and self.age > 12:
                        action = self._repr
                        self.hunger = self.max_hunger // 3
                    elif self.hunger > (self.max_hunger // 10) or self.aggr >= self.max_swarm:
                        action = self._eat
                    else:
                        action = self._move_to_swarm

                    def lifecycle():
                        action()
                        self.hunger += 1

                    self._lifecycle = lifecycle

            def _eat(self):
                target = self._hunt()

                if target is None:
                    self._move()
                else:
                    self.world.del_entity(self.world.entities[target])
                    self._move(target)
                    self.hunger = 0

            def _hunt(self):
                if self.aggr > self.max_swarm:
                    self.swarm.remove(self.coord)
                    return rnd.choice(self.swarm)

                observed = circle(*self.coord, r=self.vision_radius)
                rnd.shuffle(observed)

                for coord in observed:
                    if coord in self.world.entities.keys():
                        target_class = find_class_name(self.world.entities[coord])
                        if target_class in self.diet:
                            return coord

                return None
            
            def _move(self, target=None):
                if target is None:
                    coords = crusade(*self.coord)
                    places = [self.coord]

                    for coord in coords:
                        if not coord in self.world:
                            continue
                        elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                            places.append(coord)

                    target = rnd.choice(places)

                self.world.move_entity(self, target)
                self.coord = target

            def _move_to_swarm(self):
                target = rnd.choice(self.swarm)
                places = circle(*target, r=self.vision_radius)
                rnd.shuffle(places)

                for place in places:
                    if place in self.world.free_places:
                        self._move(place)
                        return
                
                self._move()

            def _find_pair(self):
                for coord in self.swarm:
                    if self.world.entities[coord].sex != self.sex:
                        return True
                
                return False

            def _repr(self):
                coords = crusade(*self.coord)
                place = None

                for coord in coords:
                    if not coord in self.world:
                        continue
                    elif coord in self.world.free_places or find_parent_name(self.world.entities[coord]) != find_parent_name(self):
                        place = coord
                        break

                if place is None:
                    return
                
                new_animal = self.__class__(self.world, place)
                if place in self.world.free_places:
                    self.world.add_entity(new_animal)
                else:
                    self.world.repl_entity(new_animal)

            def _find_swarm(self):
                swarm = [self.coord]
                visited = set()
                queue = [self.coord]
                visited.add(self.coord)

                while queue:
                    current_coord = queue.pop(0)
                    neighbors = circle(*current_coord, r=self.vision_radius)

                    for neighbor in neighbors:
                        if neighbor not in visited:
                            visited.add(neighbor)
                            if (neighbor in self.world.entities and 
                                find_class_name(self.world.entities[neighbor]) == find_class_name(self)):
                                swarm.append(neighbor)
                                queue.append(neighbor)

                return swarm
            
            cls._adapt = _adapt
            cls._eat = _eat
            cls._hunt = _hunt
            cls._move = _move
            cls._move_to_swarm = _move_to_swarm
            cls._find_pair = _find_pair
            cls._repr = _repr
            cls._find_swarm = _find_swarm

            cls.diet = []
            cls.sex = 'G'
            cls.aggr = 0
            cls.hunger = 0
            cls.max_hunger = 150
            cls.max_swarm = 0
            cls.max_age = 480
            cls.vision_radius = 2

        return cls

In [52]:
class Plant(Entity, metaclass=EcosystemMeta):
    def __init__(self, world, coord):
        super().__init__(world, coord)

class Demi(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "| D "
        self.active_time = ["Morning", "Afternoon"]

class Obscurite(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "| O "
        self.active_time = ["Night", "Afternoon"]

class Lumiere(Plant):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "| L "
        self.active_time = ["Morning", "Day"]

In [53]:
class Animal(Entity, metaclass=EcosystemMeta):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.sex = sex_gen()
        self.swarm = [coord]
    
class Pauvre(Animal):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "| P "
        self.active_time = ["Morning", "Day", "Afternoon"]
        self.diet = ["Lumiere"]
        self.max_swarm = 5

class Malheureux(Animal):
    def __init__(self, world, coord):
        super().__init__(world, coord)
        self.mark = "| M "
        self.active_time = ["Morning", "Afternoon"]
        self.diet = ["Demi", "Obscurite", "Pauvre"]
        self.max_swarm = 7

# Графическое представление

In [54]:
import PySimpleGUI as sg
from typing import Dict, Tuple, List
from copy import deepcopy
from dataclasses import dataclass

In [55]:
ENTITY_VISUAL = {'L': ('#FFFF00', 'square', 'Lumiere'),
                 'O': ('#0000FF', 'square', 'Obscurite'),
                 'D': ('#808080', 'square', 'Demi'),
                 'M': ('#800080', 'circle', 'Malheureux'),
                 'P': ('#FFFF00', 'circle', 'Pauvre')}

In [ ]:
@dataclass
class WorldState:
    entities: Dict[Tuple[int, int], object]
    free_places: List[Tuple[int, int]]
    plain: List[List[str]]
    time: str
    ticks: int
    stats: Dict[str, float]

def create_window(world_size):
    cell_size = 20
    canvas_size = (world_size[0] * cell_size, world_size[1] * cell_size)

    stats_layout = [
        [sg.Column([
            [sg.Text('Lumiere:'), sg.Text('0', key='-LUMIERE-')],
            [sg.Text('Obscurite:'), sg.Text('0', key='-OBSCURITE-')],
            [sg.Text('Demi:'), sg.Text('0', key='-DEMI-')],
            [sg.Text('Malheureux:'), sg.Text('0', key='-MALHEUREUX-')],
            [sg.Text('Pauvre:'), sg.Text('0', key='-PAUVRE-')]
        ], pad=(0,0)),
         sg.Column([
            [sg.Text('Всего:'), sg.Text('0', key='-TOTAL-')],
            [sg.Text('Свободно:'), sg.Text('0', key='-FREE-')]
        ], pad=(20,0))]
    ]
    
    layout = [
        [sg.Text('', key='-TIME-', size=(30, 1), font='Any 12')],
        [sg.Slider(range=(0, 23), default_value=0, orientation='h', 
                  key='-TICK-', enable_events=True, expand_x=True)],
        [sg.Graph(
            canvas_size=canvas_size,
            graph_bottom_left=(0, 0),
            graph_top_right=(world_size[0], world_size[1]),
            key='-MAP-',
            enable_events=True,
            background_color='white',
            drag_submits=False)],
        [sg.Frame('Статистика', stats_layout, expand_x=True)],
        [sg.Multiline('', key='-DETAILS-', size=(60, 5), disabled=True, expand_x=True)],
        [sg.Button('Запуск', key='-START-'), 
         sg.Button('Пауза', key='-PAUSE-', disabled=True),
         sg.Button('Шаг', key='-STEP-'),
         sg.Button('Сброс', key='-RESET-')]
    ]
    
    return sg.Window('Симулятор Экосистемы', layout, resizable=True, finalize=True)

def draw_world(graph: sg.Graph, world: World, hover_pos=None, selected_pos=None):
    graph.erase()

    for x in range(world.width + 1):
        graph.draw_line((x, 0), (x, world.height), color='black')
    for y in range(world.height + 1):
        graph.draw_line((0, y), (world.width, y), color='black')

    for coord, entity in world.entities.items():
        x, y = coord
        mark = entity.mark[2]
        if mark in ENTITY_VISUAL:
            color, shape, _ = ENTITY_VISUAL[mark]
            if shape == 'square':
                graph.draw_rectangle((x, y), (x+1, y+1), fill_color=color, line_color='black')
            else:
                size = getattr(entity, 'scale', 0.5)
                graph.draw_circle((x+0.5, y+0.5), size/2, fill_color=color, line_color='black')

    pos = hover_pos or selected_pos
    if pos and pos in world.entities:
        entity = world.entities[pos]
        if hasattr(entity, 'vision_radius'):
            x, y = pos
            vr = entity.vision_radius
            graph.draw_circle((x+0.5, y+0.5), vr, 
                             fill_color=None, line_color=ENTITY_VISUAL[entity.mark[2]][0]+'40',
                             line_width=2)

            for neighbor_coord, neighbor in world.entities.items():
                if neighbor_coord != pos and hasattr(neighbor, 'vision_radius'):
                    nx, ny = neighbor_coord
                    distance = ((x - nx)**2 + (y - ny)**2)**0.5
                    if distance <= vr:
                        graph.draw_circle((nx+0.5, ny+0.5), getattr(neighbor, 'scale', 0.5)/2,
                                        fill_color=None, line_color='red', line_width=2)
    
    if selected_pos and selected_pos in world.entities:
        x, y = selected_pos
        graph.draw_rectangle((x, y), (x+1, y+1), line_color='red', line_width=3)

def update_stats(window: sg.Window, world: World, selected_entity=None):
    counts = {'L': 0, 'O': 0, 'D': 0, 'M': 0, 'P': 0}
    
    for entity in world.entities.values():
        mark = entity.mark[2]
        counts[mark] += 1
    
    window['-LUMIERE-'].update(counts['L'])
    window['-OBSCURITE-'].update(counts['O'])
    window['-DEMI-'].update(counts['D'])
    window['-MALHEUREUX-'].update(counts['M'])
    window['-PAUVRE-'].update(counts['P'])
    window['-TOTAL-'].update(len(world.entities))
    window['-FREE-'].update(len(world.free_places))
    window['-TIME-'].update(world.cel_obj.time)
    
    if selected_entity:
        entity = selected_entity
        mark = entity.mark[2]
        details = [
            f"Тип: {ENTITY_VISUAL[mark][2]}",
            f"Координаты: {entity.coord}",
            f"Радиус обзора: {getattr(entity, 'vision_radius', 'N/A')}"
        ]
        
        if hasattr(entity, 'vision_radius'):
            neighbors = []
            x, y = entity.coord
            for coord, neighbor in world.entities.items():
                if coord != entity.coord and hasattr(neighbor, 'vision_radius'):
                    nx, ny = coord
                    distance = ((x - nx)**2 + (y - ny)**2)**0.5
                    if distance <= entity.vision_radius:
                        neighbors.append(f"{ENTITY_VISUAL[neighbor.mark[2]][2]} at {coord}")
            
            details.append(f"\nСоседи ({len(neighbors)}):")
            details.extend(neighbors[:5])
            if len(neighbors) > 5:
                details.append(f"...и ещё {len(neighbors)-5}")
        
        window['-DETAILS-'].update('\n'.join(details))
    else:
        window['-DETAILS-'].update('Выберите существо для детальной информации')

def collect_world_state(world: World) -> WorldState:
    counts = {'L': 0, 'O': 0, 'D': 0, 'M': 0, 'P': 0}
    
    for entity in world.entities.values():
        mark = entity.mark[2]
        counts[mark] += 1
    
    stats = {
        'lumiere': counts['L'],
        'obscurite': counts['O'],
        'demi': counts['D'],
        'malheureux': counts['M'],
        'pauvre': counts['P'],
    }
    
    return WorldState(
        entities=deepcopy(world.entities),
        free_places=deepcopy(world.free_places),
        plain=deepcopy(world.plain),
        time=world.cel_obj.time,
        ticks=world.cel_obj.ticks,
        stats=stats
    )

def main_game_loop(initial_entities=[], world_size=(10, 10)):
    game = GameLoop(initial_entities, world_size)
    window = create_window(world_size)
    graph = window['-MAP-']
    running = False
    selected_pos = None
    world_history = [collect_world_state(game.world)]
    hover_pos = None

    draw_world(graph, game.world, selected_pos=selected_pos)
    update_stats(window, game.world)
    
    while True:
        event, values = window.read(timeout=100 if running else None)
        
        if event == sg.WINDOW_CLOSED:
            break

        if event == '-START-':
            running = True
            window['-START-'].update(disabled=True)
            window['-PAUSE-'].update(disabled=False)
            
        elif event == '-PAUSE-':
            running = False
            window['-START-'].update(disabled=False)
            window['-PAUSE-'].update(disabled=True)
            
        elif event == '-STEP-':
            game._tick()
            world_history.append(collect_world_state(game.world))
            if len(world_history) > 24:
                world_history.pop(0)
            window['-TICK-'].update(game.world.cel_obj._get_time())
            
        elif event == '-RESET-':
            game = GameLoop(initial_entities, world_size)
            world_history = [collect_world_state(game.world)]
            selected_pos = None
            window['-TICK-'].update(0)
            
        elif event == '-TICK-':
            target_tick = int(values['-TICK-'])
            if 0 <= target_tick < len(world_history):
                state = world_history[target_tick]
                game.world.entities = deepcopy(state.entities)
                game.world.free_places = deepcopy(state.free_places)
                game.world.plain = deepcopy(state.plain)
                game.world.cel_obj.time = state.time
                game.world.cel_obj.ticks = state.ticks
                
        elif event == '-MAP-':
            x, y = values['-MAP-']
            hover_pos = (int(x), int(y))

            if values['-MAP-'][0] == 'Left':
                if hover_pos in game.world.entities:
                    selected_pos = hover_pos
                else:
                    selected_pos = None

        if hover_pos:
            draw_world(graph, game.world, hover_pos=hover_pos, selected_pos=selected_pos)
        else:
            draw_world(graph, game.world, selected_pos=selected_pos)

        selected_entity = game.world.entities.get(selected_pos) if selected_pos else None
        update_stats(window, game.world, selected_entity)

        if running:
            game._tick()
            world_history.append(collect_world_state(game.world))
            if len(world_history) > 24:
                world_history.pop(0)
            window['-TICK-'].update(game.world.cel_obj._get_time())
    
    window.close()

In [57]:
main_game_loop([(Lumiere, 10), (Obscurite, 10), (Demi, 10), (Malheureux, 5), (Pauvre, 5)], world_size=(60, 20))